# Notebook 4 : JSON et API

In [1]:
import json
import random

from io import StringIO # Pour éviter les avertissements de read_json

import pandas as pd
import requests

## Format JSON

Nous considérons deux jeux de données artificiels pour illustrer des limites du format JSON à garder à l'esprit en pratique.

In [2]:
nombres = pd.DataFrame({"Nombre": [random.random() for _ in range(5)]})
nananinf = pd.DataFrame({"Valeur": [3.14, pd.NA, float("nan"), float("inf")]})

1. Convertir `nombres` au format JSON avec la méthode `to_json` et stocker le résultat dans une variable `nombres_json`.

2. Importer `nombres_json` avec la fonction `read_json` de Pandas dans un dataframe `nombres_bis`. Comparer les objets `nombres` et `nombres_bis`.

3. Lire la documentation de `to_json` pour connaître l'option permettant de gérer (mais pas de résoudre) le problème précédent.

4. Convertir `nananinf` au format JSON avec la méthode `to_json` et stocker le résultat dans une variable `nananinf_json`. Que sont devenus `NA`, `NaN` et `inf` ?

5. Importer `nananinf_json` avec la fonction `read_json` de Pandas dans un dataframe `nananinf_bis`. Comparer les objets `nananinf` et `nananinf_bis`.

6. Reprendre les questions 4 et 5 sur l'objet `[float("nan"), float("inf")]` avec les fonctions `dumps` et `loads`. Quelle est la différence ? Lire la documentation de `dumps` pour comprendre l'option `allow_nan`.

## Iris

Nous reprenons ici le jeu de données des [Iris de Fisher](https://fr.wikipedia.org/wiki/Iris_de_Fisher) pour étudier les différentes façons d'exporter un dataframe au format JSON.

1. Charger le jeu de données dans un dataframe `iris` à partir du fichier `iris.csv`.

In [5]:
iris = pd.read_csv("/home/onyxia/work/D-pot-test/DATA/iris.csv", sep = ',')
iris

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


2. Comparer les résultats obtenus en exportant `iris` au format JSON avec `to_json` et :
- `orient="columns"`,
- `orient="index"`,
- `orient="records"`.

In [9]:
iris.to_json(orient="records")

'[{"SepalLength":5.1,"SepalWidth":3.5,"PetalLength":1.4,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":4.9,"SepalWidth":3.0,"PetalLength":1.4,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":4.7,"SepalWidth":3.2,"PetalLength":1.3,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":4.6,"SepalWidth":3.1,"PetalLength":1.5,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":5.0,"SepalWidth":3.6,"PetalLength":1.4,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":5.4,"SepalWidth":3.9,"PetalLength":1.7,"PetalWidth":0.4,"Species":"setosa"},{"SepalLength":4.6,"SepalWidth":3.4,"PetalLength":1.4,"PetalWidth":0.3,"Species":"setosa"},{"SepalLength":5.0,"SepalWidth":3.4,"PetalLength":1.5,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":4.4,"SepalWidth":2.9,"PetalLength":1.4,"PetalWidth":0.2,"Species":"setosa"},{"SepalLength":4.9,"SepalWidth":3.1,"PetalLength":1.5,"PetalWidth":0.1,"Species":"setosa"},{"SepalLength":5.4,"SepalWidth":3.7,"PetalLength":1.5,"PetalWidth":0.2,"Species":"setosa

3. Exporter `iris` dans un fichier `iris.json` au format NDJSON. Ouvrir ce fichier dans un éditeur de texte pour vérifier que chaque ligne contient un document.

4. Importer le fichier `iris.json` au format NDJSON dans un dataframe `iris2`.

## Star Wars API

Le projet SWAPI (*Star Wars API*) est une source de données sur l'univers de Star Wars. L'API fournit plusieurs jeux de données concernant les planètes, les vaisseaux, les véhicules, les personnages, les films et les espèces de la saga venue d'une galaxie très, très lointaine.

1. Utiliser la fonction Pandas `read_json` pour importer les données sur les planètes disponibles au format JSON à l'adresse [https://swapi-node.vercel.app/api/planets](https://swapi-node.vercel.app/api/planets) dans un dataframe. Est-ce que le résultat est facilement exploitable sous cette forme ?

In [11]:
# Lecture directe du JSON
url = "https://swapi-node.vercel.app/api/planets"
df = pd.read_json(url)

# Affichage
print(df.head())
print(df.columns)

   count  pages                 next  previous  \
0     60      6  /api/planets?page=2       NaN   
1     60      6  /api/planets?page=2       NaN   
2     60      6  /api/planets?page=2       NaN   
3     60      6  /api/planets?page=2       NaN   
4     60      6  /api/planets?page=2       NaN   

                                             results  
0  {'fields': {'edited': '2014-12-20T20:58:18.411...  
1  {'fields': {'edited': '2014-12-20T20:58:18.420...  
2  {'fields': {'edited': '2014-12-20T20:58:18.421...  
3  {'fields': {'edited': '2014-12-20T20:58:18.423...  
4  {'fields': {'edited': '2014-12-20T20:58:18.425...  
Index(['count', 'pages', 'next', 'previous', 'results'], dtype='object')


2. Utiliser la fonction `get` du module `requests` pour récupérer les mêmes données que dans la question précédente et vérifier le code HTTP obtenu.

In [12]:
# Requête GET
response = requests.get(url)

# Vérifier le code HTTP
print("Code HTTP :", response.status_code)

Code HTTP : 200


3. Comprendre les éléments de la réponse obtenue à la question précédente. En particulier, combien y a-t-il de planètes dans `results` et à quoi correspond `next` ?

<module 'requests' from '/opt/python/lib/python3.13/site-packages/requests/__init__.py'>

4. Écrire une boucle pour récupérer les informations de toutes les planètes disponibles dans l'API et stocker le résultat dans un dataframe `planets`.

In [17]:
url = "https://swapi-node.vercel.app/api/planets"

all_planets = []

# Boucle sur toutes les pages
while url:
    response = requests.get(url)
    
    if response.status_code != 200:
        print("Erreur :", response.status_code)
        break
    
    data = response.json()
    
    # Ajouter les planètes de la page courante
    all_planets.extend(data["results"])
    
    # Passer à la page suivante
    url = data["next"]

# Conversion en DataFrame
planets = pd.DataFrame(all_planets)

# Affichage
print(planets.head())
print("Nombre total de planètes :", len(planets))

MissingSchema: Invalid URL '/api/planets?page=2': No scheme supplied. Perhaps you meant https:///api/planets?page=2?

5. Exporter le dataframe obtenu à la question précédente dans un fichier `planets.json` au format NDJSON.